# Step 13 — Generalized difference subspace & soft threshold vote

Interactive demo of `frames/linalg/gds.py` and
`aggregators.soft_constrained`. Synthetic checks run on CPU; the GPU section
shows common/difference structure of real concept pairs.


In [ ]:
import torch

from frames.linalg.gds import (
    difference_subspace,
    gds_blend,
    gds_split,
    remove_common,
)
from frames.representations import aggregators

D = 12
g = torch.Generator().manual_seed(0)
basis, _ = torch.linalg.qr(torch.randn(D, 5, generator=g, dtype=torch.float64))
f1 = torch.cat([basis[:, :1], basis[:, 1:3]], dim=1)  # share first column
f2 = torch.cat([basis[:, :1], basis[:, 3:5]], dim=1)
frames = torch.stack([f1, f2])

common, difference, eigenvalues = gds_split(frames)
print("eigenvalues:", eigenvalues.round(decimals=3).tolist())
print("common dims:", common.shape[1], "difference dims:", difference.shape[1])

cleaned = remove_common(frames)
print("shared column zeroed:", float(cleaned[0][:, 0].norm()) < 1e-9)

blend = gds_blend(frames)
print("gds_blend orthogonal to common:", float((common.mT @ blend).abs().max()) < 1e-9)

ds = difference_subspace(f1, f2)
print("difference subspace dims (shared dropped):", ds.shape[1])

# soft threshold vote: smooth barrier vs the hard -1e4 cliff
scores = torch.tensor([[0.5, z] for z in torch.linspace(-1, 1, 9)])
soft = aggregators.soft_constrained(scores, torch.ones(2))
hard = aggregators.constrained(scores, torch.ones(2), thresholds=[0.0], penalty=-1e4)
for row, s_val, h_val in zip(scores, soft, hard):
    print(
        f"z2={float(row[1]):+.2f}  soft={float(s_val):+.3f}  hard={float(h_val):+.1f}"
    )

In [ ]:
# GPU section: common/difference structure of real concept pairs
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4", device_map="cuda:0"
)
ca = fur.get_concept_cached("happiness.n.01", 11, 3)
cb = fur.get_concept_cached("sadness.n.01", 11, 3)
stack = torch.cat(Concept._pad_to_common_rank([ca, cb])).double()
common, difference, eigenvalues = gds_split(stack)
print("rho:", float(ca.rho(cb)))
print("eigenvalues:", eigenvalues.round(decimals=2).tolist())
print(f"common dims: {common.shape[1]}, difference dims: {difference.shape[1]}")